In [ ]:

import torch, gc


torch.cuda.empty_cache()


gc.collect()


if 'model' in globals():
    del model

In [ ]:
import os
import shutil
from pathlib import Path
from huggingface_hub import login

def reset_hf_env_and_cache():

    for k in ["HF_TOKEN", "HUGGINGFACE_HUB_TOKEN", "HF_HUB_TOKEN"]:
        if k in os.environ:
            os.environ.pop(k)
            print("unset env:", k)


    paths = [
        Path("/root/.cache/huggingface/hub"),
        Path("/root/.cache/huggingface/datasets"),
    ]
    for p in paths:
        if p.exists():
            shutil.rmtree(p)
            print("removed:", p)
        else:
            print("not found:", p)

    print("Environment variables and cache cleaned.")

reset_hf_env_and_cache()
# Authenticate before running gated models if needed, for example with `huggingface-cli login`.


In [ ]:
# ==========================

# ==========================
!pip install -q "transformers>=4.44.0" datasets accelerate
!pip install sentence-transformers rouge-score

# !pip install -q sentence-transformers


In [ ]:
# ==========================

# ==========================
import re  # [ADD] for robust dash splitting
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)


# from sentence_transformers import SentenceTransformer, util
# import numpy as np


In [ ]:
# ==========================

# ==========================
dataset_name = "NifferLi/cold-chain-strawberry-sensors"
ds = load_dataset(dataset_name)
print(ds)


In [ ]:
# ==========================

# ==========================


test_splits = {name: ds[name] for name in ds.keys() if name.startswith("s")}

print("Eval splits & sizes:")
for k, v in test_splits.items():
    print(f"  {k}: {len(v)}")


In [ ]:
# ==========================

# ==========================
PROMPT_TEMPLATE = """You are an assistant that generates short, practical alerts for refrigerated-truck drivers transporting strawberries.

Your job:
- Turn a structured 1-hour risk summary into a 4-line driver message.

Hard rules:
- Use this fixed format and these exact line headers:
  Line 1: Status: ...
  Line 2: Temp pattern: ...
  Line 3: Data quality: ...
  Line 4: Action: ...
- Use simple, practical language suitable for a busy driver.
  - Short sentences.
  - No technical terms (no "respiration rate", "variance", etc.).
- Focus on:
  - what is happening in this 1-hour window,
  - and what the driver should do now.
- Map risk level to Status:
  - R0 → "Status: Normal – ..."
  - R1 → "Status: Warning – ..."
  - R2 → "Status: High risk – ..."
- Map confidence level to Data quality:
  - high confidence → "Data quality: High – ..."
  - medium confidence → "Data quality: Medium – ..."
  - low confidence → "Data quality: Low – ..."
- Map the Action line as follows, and ALWAYS include a dash (–) after the main verb:
  - For R0 (normal): use "Action: Maintain – ..." with a short instruction
    (e.g., keep current settings, drive as usual).
  - For R1 (warning): use "Action: Review – ..." with a short instruction
    (e.g., reduce long door openings, check set-point and airflow).
  - For R2 (high risk): use "Action: Act now – ..." with a short instruction
    (e.g., lower the fridge set-point, keep doors closed, check cooling unit or cargo).
  - Do NOT write "Action: ... with ...". Always use the pattern "Action: <verb> – <details>".
- Use the pattern of temperatures and variation:
  - If standard deviation is low and conditions are consistent → talk about "stable" or "steady" pattern.
  - If variation is large or locations differ a lot → talk about "uneven", "mixed", or "large differences".
- Do NOT mention internal model details or equations.
- Do NOT output the summary again. Only output the 4 driver lines.
- Output EXACTLY 4 lines and NOTHING else.
- Do NOT write any labels like "Answer:", "Response:", or explanations.
- Do NOT use JSON, curly braces {{ }}, or code blocks.
- Do NOT repeat the summary.
- Do NOT invent new numbers or time durations that are not clearly in the summary.
- If you mention durations, always use minutes (e.g., "30 minutes"), never seconds.

Style rules:
- Prefer qualitative wording like "long time above 4°C" or
  "short period below 0°C" instead of exact counts of minutes.
- Only mention exact durations (e.g. "30 minutes") if that number
  is clearly written in the summary.
- Do NOT use patterns like "60 minutes above 4°C and 0 minutes below 0°C".
- For warm severe cases (R2, temperatures clearly above 4°C),
  prefer phrases like:
  - "long time above 4°C"
  - "strong warm peak"
  - "moderate temperature variation"
  - "large temperature swings"
  instead of repeating numeric statistics.

Here are examples:

Example 1 – R0, low risk, stable and in-range, high confidence
Summary:
In this 1-hour window, the assessed risk level is R0 (low). All measured temperatures stay between 0°C and 4°C with only very small variation, and there are no excursions into freezing or excessive warming. This pattern matches the recommended storage range for strawberries and should support normal shelf-life. Sensor coverage is complete and consistent, so the assessment is made with high confidence and the temperature behaviour is stable.

Driver message:
Status: Normal – safe and stable temperature.
Temp pattern: In-range – temperatures stayed between 0–4°C with only small variation.
Data quality: High – sensors give a consistent and complete cold pattern.
Action: Maintain – keep current fridge settings and drive as usual.


Example 2 – R1, moderate risk, short deviations, high confidence
Summary:
In this 1-hour window, the assessed risk level is R1 (moderate). Temperatures stay close to the 0–4°C target range but drift slightly outside it for short periods, for example briefly above 4°C or just below 0°C. These deviations are not long enough to reach severe-risk conditions, but they may cause quality loss if repeated many times. Temperature variation is moderate but not extreme, and all sensors report normally, so the assessment is made with high confidence.

Driver message:
Status: Warning – minor temperature deviation.
Temp pattern: Mixed – short time just outside 0–4°C with moderate variation.
Data quality: High – sensors show the pattern clearly.
Action: Review – reduce long door openings and confirm that set-point and airflow are correct.


Example 3 – R2, severe high temperature, long time warm, stable warm pattern, high confidence
Summary:
In this 1-hour window, the assessed risk level is R2 (severe). Temperatures are well above 4°C for almost the entire hour and there is at least one strong warm peak clearly above 10°C. The mean temperature is high and the variation is not large, so the load is consistently too warm rather than just briefly spiking. This warm exposure can strongly shorten the remaining shelf-life of strawberries. Sensor coverage is high and consistent across locations, so the assessment and the warm pattern are both considered reliable.

Driver message:
Status: High risk – cargo too warm.
Temp pattern: Over-warm – almost the whole hour well above 4°C with a strong warm peak and stable warm pattern.
Data quality: High – multiple sensors agree on this warm condition.
Action: Act now – lower the fridge set-point, keep doors closed, and check the cooling unit and airflow.


Example 4 – R2, severe high temperature, uneven and with missing data, medium confidence
Summary:
In this 1-hour window, the assessed risk level is R2 (severe). Temperatures are above 4°C for a long time and there is a strong warm peak above 10°C, indicating that at least part of the load is over-warm. However, some sensor readings are missing or incomplete, and the variation between locations is quite large. This means parts of the cargo may be much warmer than others, but the exact distribution is uncertain. Because of the missing data and uneven pattern, the assessment is made with only medium confidence.

Driver message:
Status: High risk – cargo likely too warm.
Temp pattern: Over-warm and uneven – long time above 4°C with a strong warm peak and large differences inside the load.
Data quality: Medium – some sensor readings are missing or incomplete.
Action: Act now – lower the set-point, keep doors closed, and, if possible, check the cargo condition manually.


Example 5 – R2, severe risk, warm for too long without extreme peak, high confidence
Summary:
In this 1-hour window, the assessed risk level is R2 (severe). Temperatures stay above 4°C for around half an hour, while the rest of the period is close to the lower limit of the recommended range. Even without an extreme peak above 10°C, this prolonged moderate warming can still accelerate quality loss, especially if similar episodes repeat during the trip. Temperature variation is moderate and sensor readings are complete across time and locations, so the warm period is clearly visible and the assessment is made with high confidence.

Driver message:
Status: High risk – cargo warm for too long.
Temp pattern: Over-warm – about half an hour above 4°C and the rest near the lower limit, with moderate variation.
Data quality: High – sensors show a clear warm period.
Action: Act now – shorten door-opening times, check fridge capacity, and consider a slightly lower set-point.


Example 6 – R2, severe risk, long period near/below 0°C, high confidence
Summary:
In this 1-hour window, the assessed risk level is R2 (severe). The maximum temperature stays at or below 4°C, but minimum values drop close to or slightly below 0°C, and the cargo spends around 30 minutes at or below 0°C. This over-cooling near the freezing point can cause chilling injury or partial freezing if it occurs repeatedly. The temperature pattern is fairly stable and cold, and all sensors report consistently, so the assessment is made with high confidence.

Driver message:
Status: High risk – cargo too cold.
Temp pattern: Over-cold – around 30 minutes close to or below 0°C with a stable cold pattern.
Data quality: High – sensors show a consistent cold trend.
Action: Act now – raise the set-point slightly, avoid very strong cooling, and watch the coldest positions.


Example 7 – R2, severe risk, possible freezing damage, high confidence
Summary:
In this 1-hour window, the assessed risk level is R2 (severe). Temperatures are below 0°C for most of the hour, and the coldest locations drop to around or below −1°C. This cold exposure is close to or beyond the freezing point of strawberries and can cause irreversible freezing damage, such as darkening, watery texture, and collapse after re-warming. The cold pattern is steady and several sensors agree on this behaviour, so the assessment is made with high confidence.

Driver message:
Status: High risk – possible freezing damage.
Temp pattern: Freezing – long time below 0°C with the coldest spots near or below −1°C and a steady cold pattern.
Data quality: High – several sensors confirm this freezing behaviour.
Action: Act now – increase the set-point, avoid strong cooling, and flag this batch for a quality check at destination.


----------------
Now a new case:
Summary:
{PUT_SUMMARY_TEXT_HERE}

Please generate ONLY the 4-line driver message in the required format:
Line 1: Status: ...
Line 2: Temp pattern: ...
Line 3: Data quality: ...
Line 4: Action: ..."""

# def format_example(example):
#     summary = example["summary_text"]
#     driver = example["driver_message"]

#     if driver is None:
#         driver = ""

#     prompt = PROMPT_TEMPLATE.format(PUT_SUMMARY_TEXT_HERE=summary)
#     text = prompt + "\n" + driver

#     return {
#         "text": text,
#         "prompt": prompt,
#         "target_driver_message": driver,
#     }


# train_ds_fmt = train_ds.map(format_example)
# print(train_ds_fmt[0]["text"][:800])

In [ ]:
# ==========================

# ==========================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=False,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id

print("Model and tokenizer loaded:", model_name)


In [ ]:
# ==========================

# ==========================




In [ ]:
# ==========================

# ==========================

def clean_driver_message(raw_text: str) -> str:
    """Public-release documentation comment."""
    required_prefixes = [
        "Status:",
        "Temp pattern:",
        "Data quality:",
        "Action:",
    ]
    collected = {p: None for p in required_prefixes}

    for line in raw_text.splitlines():
        line = line.strip()
        if not line:
            continue

        for p in required_prefixes:
            if collected[p] is None and line.startswith(p):
                collected[p] = line
                break

        if all(collected[p] is not None for p in required_prefixes):
            break

    if all(collected[p] is not None for p in required_prefixes):
        lines_ordered = [collected[p] for p in required_prefixes]
        return "\n".join(lines_ordered)
    else:
        return raw_text.strip()


def generate_driver_message(summary: str, max_new_tokens: int = 160) -> str:
    """Public-release documentation comment."""

    user_prompt = PROMPT_TEMPLATE.format(PUT_SUMMARY_TEXT_HERE=summary)

    messages = [
        {
            "role": "user",
            "content": user_prompt,
        }
    ]


    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)


    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )


    input_len = inputs.shape[1]
    gen_ids = outputs[0][input_len:]
    gen = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


    gen_clean = clean_driver_message(gen)
    return gen_clean


_DASH_SPLIT_RE = re.compile(r"\s*[–—-]\s*")

def _main_label_before_dash(content: str) -> str:
    parts = _DASH_SPLIT_RE.split(content, maxsplit=1)
    return parts[0].strip() if parts else content.strip()
def parse_driver_message_slots(msg: str):
    """Public-release documentation comment."""
    slots = {
        "risk_level": None,
        "temp_pattern": None,
        "data_quality": None,
        "action": "",
    }
    if not isinstance(msg, str):
        return slots

    pattern = (
        r"(?is)Status:\s*(.*?)\s*"
        r"Temp pattern:\s*(.*?)\s*"
        r"Data quality:\s*(.*?)\s*"
        r"Action:\s*(.*)"
    )
    m = re.search(pattern, msg)
    if not m:
        return slots

    r_text, t_text, d_text, a_text = m.groups()

    def _get_main_label(text: str):
        text = (text or "").strip().replace("\n", " ")
        if not text:
            return None

        main = _main_label_before_dash(text)
        return main or None

    slots["risk_level"]   = _get_main_label(r_text)
    slots["temp_pattern"] = _get_main_label(t_text)
    slots["data_quality"] = _get_main_label(d_text)
    slots["action"]       = (a_text or "").strip().replace("\n", " ")

    return slots
"""Public-release documentation comment."""

Automated evaluation over each split reports format compliance, exact match against the gold driver message, token-level text overlap, ROUGE-L, and embedding-based semantic similarity. The notebook also prints a small number of examples for manual inspection, including the input summary, gold driver message, and cleaned model output.


In [ ]:
# ==========================


# ==========================

from tqdm.auto import tqdm
from typing import List, Optional, Iterable, Set, Dict, Any

import torch
import torch.nn.functional as F



# !pip -q install sentence-transformers
from sentence_transformers import SentenceTransformer


# !pip -q install rouge-score
from rouge_score import rouge_scorer


# --------------------------

# --------------------------
def load_embedder(
    model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
    device: Optional[str] = None,
):
    """Public-release documentation comment."""
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    embedder = SentenceTransformer(model_name, device=device)
    return embedder



EMBEDDER = load_embedder()


# --------------------------

# --------------------------
def check_format(msg: str) -> bool:
    """Public-release documentation comment."""
    if not isinstance(msg, str):
        return False


    pattern = (
        r"(?is)Status:\s*(.*?)\s*"
        r"Temp pattern:\s*(.*?)\s*"
        r"Data quality:\s*(.*?)\s*"
        r"Action:\s*(.*)"
    )
    return bool(re.search(pattern, msg))
# --------------------------

# --------------------------
def precision_recall_f1(tp: int, fp: int, fn: int):
    """Public-release documentation comment."""
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    if precision + recall > 0:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = 0.0
    return precision, recall, f1


# --------------------------

# --------------------------
def compute_single_label_metrics(
    gold_labels: List[Optional[str]],
    pred_labels: List[Optional[str]],
) -> Dict[str, Any]:
    """Public-release documentation comment."""
    assert len(gold_labels) == len(pred_labels)

    n = 0
    correct = 0
    tp = fp = fn = 0

    for g, p in zip(gold_labels, pred_labels):
        if g is None or g == "":
            continue

        n += 1

        if p is None or p == "":
            fn += 1
            continue

        if p == g:
            correct += 1
            tp += 1
        else:
            fp += 1
            fn += 1

    accuracy = correct / n if n > 0 else 0.0
    precision, recall, f1 = precision_recall_f1(tp, fp, fn)

    return {
        "n": n,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


# --------------------------

# --------------------------
def normalize_single_label(label: Optional[str]) -> Optional[str]:
    """Public-release documentation comment."""
    if label is None:
        return None

    s = str(label).strip()
    if not s:
        return None


    s = s.replace("—", "-").replace("–", "-")


    s = s.strip().strip(".").strip("。").strip(";").strip("；").strip(":")


    s = " ".join(s.split())


    canon_map = {
        # Status
        "high risk": "High risk",
        "warning": "Warning",
        "normal": "Normal",

        # Data quality
        "high": "High",
        "medium": "Medium",
        "low": "Low",


        "in range": "In-range",
        "in-range": "In-range",
        "slight drift": "Slight drift",
        "over warm": "Over-warm",
        "over-warm": "Over-warm",
        "over cold": "Over-cold",
        "over-cold": "Over-cold",
        "freezing": "Freezing",
    }

    key = s.lower()
    return canon_map.get(key, s)


# --------------------------

# --------------------------
def compute_token_metrics(
    gold_texts: List[str],
    pred_texts: List[str],
    tokenizer,
) -> Dict[str, Any]:
    """Public-release documentation comment."""
    assert len(gold_texts) == len(pred_texts)

    tp = fp = fn = 0
    total_jaccard = 0.0
    n_pairs = 0

    for g, p in zip(gold_texts, pred_texts):
        g = g or ""
        p = p or ""
        g_tokens = set(tokenizer.tokenize(g))
        p_tokens = set(tokenizer.tokenize(p))

        if not g_tokens and not p_tokens:
            continue

        n_pairs += 1
        inter = len(g_tokens & p_tokens)
        union = len(g_tokens | p_tokens)

        tp += inter
        fp += len(p_tokens - g_tokens)
        fn += len(g_tokens - p_tokens)

        if union > 0:
            total_jaccard += inter / union

    precision, recall, f1 = precision_recall_f1(tp, fp, fn)
    avg_jaccard = total_jaccard / n_pairs if n_pairs > 0 else 0.0

    return {
        "n": n_pairs,
        "jaccard": avg_jaccard,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


# --------------------------

# --------------------------
def compute_multilabel_metrics(
    gold_sets: List[Iterable[str]],
    pred_sets: List[Iterable[str]],
) -> Dict[str, Any]:
    """Public-release documentation comment."""
    assert len(gold_sets) == len(pred_sets)

    tp = fp = fn = 0
    total_jaccard = 0.0
    n_pairs = 0

    for g, p in zip(gold_sets, pred_sets):
        g_set: Set[str] = set(g or [])
        p_set: Set[str] = set(p or [])

        if not g_set and not p_set:
            continue

        n_pairs += 1
        inter = len(g_set & p_set)
        union = len(g_set | p_set)

        tp += inter
        fp += len(p_set - g_set)
        fn += len(g_set - p_set)

        if union > 0:
            total_jaccard += inter / union

    precision, recall, f1 = precision_recall_f1(tp, fp, fn)
    avg_jaccard = total_jaccard / n_pairs if n_pairs > 0 else 0.0

    return {
        "n": n_pairs,
        "jaccard": avg_jaccard,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


# --------------------------

# --------------------------
def extract_line_content(msg: str, prefix: str) -> str:
    """Public-release documentation comment."""
    if not isinstance(msg, str):
        return ""

    order = ["Status:", "Temp pattern:", "Data quality:", "Action:"]
    if prefix not in order:

        for line in msg.splitlines():
            line = line.strip()
            if line.startswith(prefix):
                return line[len(prefix):].strip()
        return ""

    idx = order.index(prefix)
    if idx < len(order) - 1:
        nxt = order[idx + 1]
        pattern = rf"(?is){re.escape(prefix)}\s*(.*?)\s*{re.escape(nxt)}"
        m = re.search(pattern, msg)
        return (m.group(1).strip().replace("\n", " ")) if m else ""
    else:
        pattern = rf"(?is){re.escape(prefix)}\s*(.*)"
        m = re.search(pattern, msg)
        return (m.group(1).strip().replace("\n", " ")) if m else ""
def compute_embedding_metrics_st(
    gold_texts: List[str],
    pred_texts: List[str],
    embedder: SentenceTransformer,
    batch_size: int = 32,
) -> Dict[str, Any]:
    """Public-release documentation comment."""
    assert len(gold_texts) == len(pred_texts)

    idx = []
    g_list = []
    p_list = []

    for i, (g, p) in enumerate(zip(gold_texts, pred_texts)):
        g = (g or "").strip()
        p = (p or "").strip()
        if not g or not p:
            continue
        idx.append(i)
        g_list.append(g)
        p_list.append(p)

    if not idx:
        return {"n": 0, "cosine": 0.0}

    g_emb = embedder.encode(
        g_list,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )
    p_emb = embedder.encode(
        p_list,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    sims = (g_emb * p_emb).sum(dim=1)
    mean_sim = sims.mean().item() if sims.numel() > 0 else 0.0

    return {
        "n": len(idx),
        "cosine": mean_sim
    }


# --------------------------

# --------------------------
def compute_rouge_l_metrics(
    gold_texts: List[str],
    pred_texts: List[str],
) -> Dict[str, Any]:
    """Public-release documentation comment."""
    assert len(gold_texts) == len(pred_texts)

    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

    n = 0
    sum_p = sum_r = sum_f = 0.0

    for g, p in zip(gold_texts, pred_texts):
        g = (g or "").strip()
        p = (p or "").strip()
        if not g and not p:
            continue

        scores = scorer.score(g, p)["rougeL"]
        sum_p += scores.precision
        sum_r += scores.recall
        sum_f += scores.fmeasure
        n += 1

    if n == 0:
        return {"n": 0, "precision": 0.0, "recall": 0.0, "f1": 0.0}

    return {
        "n": n,
        "precision": sum_p / n,
        "recall":    sum_r / n,
        "f1":        sum_f / n,
    }


# --------------------------

# --------------------------
def evaluate_split(
    split_ds,
    split_name: str,
    max_samples: int = 50,
    print_examples: int = 5,
    embedder: SentenceTransformer = EMBEDDER,
):
    """Public-release documentation comment."""
    n_total = len(split_ds)
    n_eval = n_total if max_samples is None else min(n_total, max_samples)

    format_ok_count = 0


    gold_risk, pred_risk = [], []
    gold_temp, pred_temp = [], []
    gold_dq,   pred_dq   = [], []
    gold_action, pred_action = [], []


    gold_status_text, pred_status_text = [], []
    gold_temp_text,   pred_temp_text   = [], []
    gold_dq_text,     pred_dq_text     = [], []
    gold_action_text, pred_action_text = [], []


    gold_msg_text, pred_msg_text = [], []

    print(f"\n\n================= Evaluation split: {split_name} =================")
    print(f"Total samples: {n_total}, evaluated samples: {n_eval}")
    print("------------------------------------------------------------")

    for i in tqdm(range(n_eval), desc=f"Evaluating {split_name}", ncols=80):
        example = split_ds[i]
        summary = example.get("summary_text", "") or ""
        gold_msg = (example.get("driver_message", "") or "").strip()


        # pred_msg = generate_driver_message(summary).strip()


        if "System Notification: Low confidence" in summary:
            pred_msg = (
                "Status: Unknown\n"
                "Temp pattern: Insufficient data for pattern analysis\n"
                "Data quality: Low (System Warning)\n"
                "Action: Manual verification required (Check sensors)"
            )
        else:

            pred_msg = generate_driver_message(summary).strip()



        if check_format(pred_msg):
            format_ok_count += 1


        gold_slots = parse_driver_message_slots(gold_msg)
        pred_slots = parse_driver_message_slots(pred_msg)

        gold_risk.append(gold_slots["risk_level"])
        pred_risk.append(pred_slots["risk_level"])

        gold_temp.append(gold_slots["temp_pattern"])
        pred_temp.append(pred_slots["temp_pattern"])

        gold_dq.append(gold_slots["data_quality"])
        pred_dq.append(pred_slots["data_quality"])

        gold_action.append(gold_slots["action"])
        pred_action.append(pred_slots["action"])


        gold_status_text.append(extract_line_content(gold_msg, "Status:"))
        pred_status_text.append(extract_line_content(pred_msg, "Status:"))

        gold_temp_text.append(extract_line_content(gold_msg, "Temp pattern:"))
        pred_temp_text.append(extract_line_content(pred_msg, "Temp pattern:"))

        gold_dq_text.append(extract_line_content(gold_msg, "Data quality:"))
        pred_dq_text.append(extract_line_content(pred_msg, "Data quality:"))

        gold_action_text.append(extract_line_content(gold_msg, "Action:"))
        pred_action_text.append(extract_line_content(pred_msg, "Action:"))


        gold_msg_text.append(gold_msg.replace("\n", " ").strip())
        pred_msg_text.append(pred_msg.replace("\n", " ").strip())


        if i < print_examples:
            print(f"\n--- Sample index: {i} ---")
            print("[Summary_text]")
            print(summary[:200] + ("..." if len(summary) > 200 else ""))

            print("\n[Gold driver_message]")
            print(gold_msg)

            print("\n[Generated driver_message, cleaned]")
            print(pred_msg)
            print("------------------------------------------------------------")


    format_ratio = format_ok_count / n_eval if n_eval > 0 else 0.0

    risk_metrics = compute_single_label_metrics(
        [normalize_single_label(x) for x in gold_risk],
        [normalize_single_label(x) for x in pred_risk],
    )

    temp_metrics = compute_single_label_metrics(
        [normalize_single_label(x) for x in gold_temp],
        [normalize_single_label(x) for x in pred_temp],
    )

    dq_metrics = compute_single_label_metrics(
        [normalize_single_label(x) for x in gold_dq],
        [normalize_single_label(x) for x in pred_dq],
    )

    action_metrics = compute_token_metrics(gold_action, pred_action, tokenizer)
    action_rouge   = compute_rouge_l_metrics(gold_action, pred_action)


    status_emb = compute_embedding_metrics_st(gold_status_text, pred_status_text, embedder)
    temp_emb   = compute_embedding_metrics_st(gold_temp_text,   pred_temp_text,   embedder)
    dq_emb     = compute_embedding_metrics_st(gold_dq_text,     pred_dq_text,     embedder)
    action_emb = compute_embedding_metrics_st(gold_action_text, pred_action_text, embedder)

    # Overall slot-level embedding
    emb_slot_overall_sum = 0.0
    emb_slot_overall_n = 0
    for m in [status_emb, temp_emb, dq_emb, action_emb]:
        if m["n"] > 0:
            emb_slot_overall_sum += m["cosine"]
            emb_slot_overall_n += 1
    emb_slot_overall = emb_slot_overall_sum / emb_slot_overall_n if emb_slot_overall_n > 0 else 0.0

    # Message-level embedding
    msg_emb = compute_embedding_metrics_st(gold_msg_text, pred_msg_text, embedder)


    print("\n>>> Overall message format <<<")
    print(f"Fully valid format: {format_ok_count}/{n_eval} "
          f"= {format_ratio:.1%}")

    print("\n>>> Slot-level single-label classification metrics <<<")

    def print_cls_slot(name: str, m: dict):
        print(f"\n[{name}]")
        print(f"Sample size n  : {m['n']}")
        print(f"Accuracy       : {m['accuracy']:.3f}")
        print(f"Precision (P)  : {m['precision']:.3f}")
        print(f"Recall    (R)  : {m['recall']:.3f}")
        print(f"F1-score (F1)  : {m['f1']:.3f}")

    print_cls_slot("risk_level (Status line)", risk_metrics)
    print_cls_slot("temp_pattern (Temp pattern line)", temp_metrics)
    print_cls_slot("data_quality (Data quality line)", dq_metrics)

    print("\n[action] short-text generation -> token-level metrics")
    print(f"Sample size n       : {action_metrics['n']}")
    print(f"Jaccard (token-set): {action_metrics['jaccard']:.3f}")
    print(f"token-Precision (P): {action_metrics['precision']:.3f}")
    print(f"token-Recall    (R): {action_metrics['recall']:.3f}")
    print(f"token-F1         F1: {action_metrics['f1']:.3f}")

    print("\n[action] short-text generation -> ROUGE-L metrics")
    print(f"Sample size n       : {action_rouge['n']}")
    print(f"ROUGE-L Precision  : {action_rouge['precision']:.3f}")
    print(f"ROUGE-L Recall     : {action_rouge['recall']:.3f}")
    print(f"ROUGE-L F1         : {action_rouge['f1']:.3f}")

    print("\n>>> Embedding semantic similarity (cosine, SentenceTransformer) <<<")
    print(f"[Status]       n={status_emb['n']}, cosine={status_emb['cosine']:.3f}")
    print(f"[Temp pattern] n={temp_emb['n']},   cosine={temp_emb['cosine']:.3f}")
    print(f"[Data quality] n={dq_emb['n']},     cosine={dq_emb['cosine']:.3f}")
    print(f"[Action]       n={action_emb['n']}, cosine={action_emb['cosine']:.3f}")

    print("\n>>> Embedding summary <<<")
    print(f"Overall slot-embedding (avg of 4 slots): {emb_slot_overall:.3f}")
    print(f"Message-level embedding: n={msg_emb['n']}, cosine={msg_emb['cosine']:.3f}")

    print("\n============================================================")


# ==========================================

# ==========================================
def has_text(example):
    s = example.get("summary_text")
    m = example.get("driver_message")
    return (s is not None and len(str(s).strip()) > 0) and (m is not None and len(str(m).strip()) > 0)

test_splits_nonempty = {
    name: split.filter(has_text)
    for name, split in test_splits.items()
}

# ==========================

# ==========================


# ==========================

# ==========================
target_splits = ["s1", "s2", "s4", "s5", "s6"]
splits_to_run = {k: v for k, v in test_splits_nonempty.items() if k in target_splits}

print(f"Splits to run: {list(splits_to_run.keys())}")
for split_name, split_ds in splits_to_run.items():
    evaluate_split(
        split_ds,
        split_name=split_name,
        max_samples=None,
        print_examples=5,
        embedder=EMBEDDER,
    )
